In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm.auto import tqdm

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '2_Propensities'))
import MF_class as MF

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

### 0. Choose Dataset

In [2]:
datasets = ['ml-1m', 'steam', 'goodreads']
DATASET = datasets[2]

In [3]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET

# 1. Load Data

In [4]:
# Load train and test data for MF
train = pd.read_csv(data_path / 'train.csv')
test = pd.read_csv(data_path / 'test.csv')
with open(data_path / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)

n_users = train['user_id'].nunique()
n_items = train['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

test_users = test['user_id'].unique()
all_users = train['user_id'].unique()
train_users = np.setdiff1d(all_users, test_users)

# Load chosen pairs and convert to item ids
with open(base_artifacts / 'Datasets' / 'Processed' / DATASET / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)
reverse_item_dict = {v: k for k, v in item_dict.items()}

with open(base_artifacts / 'Chosen_Pairs' / f'{DATASET}_chosen_pairs.pkl', 'rb') as f:
    chosen_pairs = pickle.load(f)
pairs_ids = [(reverse_item_dict[pair[0]], reverse_item_dict[pair[1]]) for pair in chosen_pairs]

Number of users: 7801, Number of items: 6384


# 2. Create Training and Validation Sets for Outcome Model

In [5]:
def get_timestamp_dict(data, users):

    subset = data[data['user_id'].isin(users)]
    pos = (
        subset[subset['interaction'] == 1]
        .groupby(['user_id', 'item_id'])['timestamp']
        .first()
    )

    timestamp_dict = {user_id : user_pos.droplevel(0).to_dict() for user_id, user_pos in tqdm(pos.groupby(level=0))}
    for user_id in users:
        if user_id not in timestamp_dict:
            timestamp_dict[user_id] = {}

    return timestamp_dict

def get_om_data(data, users, item_pairs_id, users_per_pair, random_state=42): 
    rng = np.random.default_rng(seed=random_state)
    len_users = len(users) 
    
    print("Constructing timestamp dictionary...")
    timestamp_dict = get_timestamp_dict(data, users)
    
    print("Generating OM training data...")
    rows = []
    for (i, j) in tqdm(item_pairs_id):
        for _ in range(users_per_pair):
            user = users[rng.integers(len_users)]
            pos_items_dict = timestamp_dict[user]

            x, y = 0, 0
            if i in pos_items_dict:
                if j in pos_items_dict:
                    if pos_items_dict[i] < pos_items_dict[j]:
                        x, y = 1, 1
                    else:
                        continue
                else:
                    x = 1
            elif j in pos_items_dict:
                y = 1
                    
            rows.append((user, i, j, x, y)) 
    
    return pd.DataFrame(rows, columns=["u", "i", "j", "x", "y"])

In [6]:
users_per_pair = 300

om_train_data = get_om_data(
    data=train, 
    users=train_users, 
    item_pairs_id=pairs_ids, 
    users_per_pair=users_per_pair, 
    random_state=42)

om_test_data  = get_om_data(
    data=test,  
    users=test_users, 
    item_pairs_id=pairs_ids, 
    users_per_pair=users_per_pair, 
    random_state=42)

Constructing timestamp dictionary...


  0%|          | 0/3901 [00:00<?, ?it/s]

Generating OM training data...


  0%|          | 0/10000 [00:00<?, ?it/s]

Constructing timestamp dictionary...


  0%|          | 0/3895 [00:00<?, ?it/s]

Generating OM training data...


  0%|          | 0/10000 [00:00<?, ?it/s]

# 3. Save Datasets

In [7]:
om_train_data.to_csv(data_path / 'om_train.csv', index=False)
om_test_data.to_csv(data_path / 'om_test.csv', index=False)